# Tutorial — fitting photometry, spectroscopy and emission lines jointly

CERIDWEN treats the three observation types uniformly: each is a small container that projects the model spectrum onto its own data space, and the joint likelihood is the sum of their χ² contributions. You can fit any subset, or all three together, with no change to the model or sampler.

This notebook is a **template** — replace the placeholder `my_*` arrays with your own data. For a fully runnable photometry-only demo (it generates its own mock data), see `examples/quickstart.py`.

> **Read first:** `Z` is log10 *absolute* metallicity (grid ≈ [-4, -1.4]); `lookback_time` index 0 is *today*. FSPS + `$SPS_HOME` must be set up — emission lines and nebular continuum read the CLOUDY grids from FSPS. Requires Python ≥ 3.11.

## 1. Imports and the SSP grid

In [ ]:
import jax, jax.numpy as jnp
import numpy as np

from ceridwen import SSPData, CSPBasis, SedModel, fitSED
from ceridwen.observation import Photometry, Spectrum, Lines
from ceridwen.priors import Uniform, ClippedNormal, StudentT
from ceridwen.model import logsfr_ratios_to_sfh

# Build once with SSPData.from_fsps(imf_type=1, save_to='ssp_data.h5'); then reload:
ssp = SSPData.load('ssp_data.h5')

ZRED = 0.5   # spectroscopic redshift of the galaxy

## 2. Forward model

Emission lines and nebular continuum require `add_neb=True`. Defaults give a Charlot & Fall power-law birth-cloud plus a Kriek & Conroy diffuse screen; `sps_home` defaults to `$SPS_HOME`.

In [ ]:
csp = CSPBasis(
    ssp,
    add_dust=True, add_diffuse_dust=True,   # birth-cloud (power law) + diffuse (Kriek & Conroy)
    add_neb=True,                           # nebular continuum + lines (needs $SPS_HOME)
    add_igm=True,                           # Madau (1995), auto-scales with zred
    sfh_interp='step',
    verbose=False,
)

## 3. (a) Photometry

Broadband fluxes in **AB maggies**. If your catalogue is in nJy (common for JWST), convert: `maggies = flux_nJy * 1e-9 / 3631`. A small error floor (e.g. 5%) stabilises the fit. Photometry captures the full aperture, so it sees the intrinsic line + continuum flux.

In [ ]:
FILTERS = ['jwst_f090w','jwst_f115w','jwst_f150w','jwst_f200w',
           'jwst_f277w','jwst_f356w','jwst_f444w']

flux_nJy = my_phot_nJy            # shape (n_filters,), your data
unc_nJy  = my_phot_unc_nJy
unc_nJy  = np.where(unc_nJy / flux_nJy > 0.05, unc_nJy, 0.05 * flux_nJy)   # 5% floor

phot = Photometry(
    filters=FILTERS,
    flux=jnp.asarray(flux_nJy) * 1e-9 / 3631.0,        # nJy -> AB maggies
    uncertainty=jnp.asarray(unc_nJy) * 1e-9 / 3631.0,
    name='phot',                                        # key this observation reports under
)

## 4. (b) Spectrum

Pass the **rest-frame, vacuum** wavelength grid (Å) and flux in `F_ν`. `resolution` + `smoothtype` apply instrumental broadening. Mask the emission lines out of the *continuum* spectrum so they are not double-counted against the `Lines` object below.

In [ ]:
spec = Spectrum(
    wavelength=my_rest_wave_aa,     # Å, vacuum, rest-frame
    flux=my_spec_fnu,               # F_nu per pixel
    uncertainty=my_spec_unc,
    resolution=120.0,               # unit set by smoothtype
    smoothtype='vel',               # 'vel' (km/s) | 'R' | 'lambda' (Aa) | 'lsf'
    inres=0.0,                      # model-library intrinsic resolution to deconvolve
    noise_floor=0.01,               # 1% multiplicative calibration floor (optional)
    name='spec',
)

# Mask known lines from the continuum fit (centres redshifted internally):
spec.mask_lines([4861.3, 5006.8, 6562.8], dv=500.0, zred=ZRED)

## 5. (c) Emission lines

Integrated line fluxes. `line_ind` are **1-based** indices into FSPS's `emlines_info.dat`; `wavelength` are vacuum rest wavelengths (Å). Catalogue fluxes are often quoted in units of 10⁻²⁰ erg s⁻¹ cm⁻², so scale to absolute CGS to match the model.

In [ ]:
LINE_UNIT = 1.0e-20   # erg s^-1 cm^-2 per catalogue unit (adjust to your catalogue!)

lines = Lines(
    line_ind=[59, 62, 63, 71, 72],                          # Hbeta, [OIII]4959/5007, Halpha, [NII]6583
    line_names=['Hbeta','[OIII]4959','[OIII]5007','Halpha','[NII]6583'],
    wavelength=[4861.3, 4958.9, 5006.8, 6562.8, 6583.4],    # Aa, vacuum rest
    flux=np.asarray(my_line_flux) * LINE_UNIT,
    uncertainty=np.asarray(my_line_unc) * LINE_UNIT,
    name='lines',
)

## 6. Priors, transforms and the model

Collect the observations into a list (any subset is fine). The non-parametric SFH is sampled as `logsfr_ratios` and turned into the per-bin `sfh` by a **registered transform** — this step is required. `logmass` and `logsfr_ratios` are seeded via `free_param_init`. Fixed knobs (the birth-cloud slope `alpha_pow`, the aperture correction `eline_scaling`) go into `theta_init`.

In [ ]:
observations = [phot, spec, lines]

sfh_times_yr = np.array(csp.sfh_times)
def logsfr_to_sfh(free_theta, _t=sfh_times_yr):
    return logsfr_ratios_to_sfh(free_theta['logsfr_ratios'], sfh_times_yr=_t)

priors = {
    # Stellar population. Z is log10 ABSOLUTE metallicity (grid ~[-4, -1.4]).
    'Z':                  ClippedNormal(mean=-2.0, sigma=0.5, low=-4.0, high=-1.4),
    'logmass':            Uniform(low=7.0, high=12.5),
    'logsfr_ratios':      StudentT(df=2.0, mean=0.0, scale=0.3),
    # Dust.
    'diffuse_tau_kc':     ClippedNormal(mean=0.3, sigma=1.0, low=0.0, high=4.0),
    'diffuse_dust_index': Uniform(low=-1.0, high=0.4),
    'tau_pow':            ClippedNormal(mean=0.3, sigma=0.5, low=0.0, high=4.0),
    # Nebular (required for the lines / nebular continuum).
    'gas_logz':           Uniform(low=-2.0, high=0.5),
    'gas_logu':           Uniform(low=-4.0, high=-1.0),
    # Aperture correction applied to the slit spectrum + lines.
    'eline_scaling':      Uniform(low=0.1, high=2.0),
}

N_RATIOS = 4   # number of SFH bins - 1
model = SedModel(
    csp,
    observations=observations,
    priors=priors,
    transforms={'sfh': logsfr_to_sfh},                 # REQUIRED for logsfr_ratios
    free_param_init={'logsfr_ratios': jnp.zeros(N_RATIOS),
                     'logmass': jnp.array([10.0])},
    zred=ZRED,
)

# Fixed knobs not sampled: birth-cloud slope, and the aperture-correction init.
model.theta_init['alpha_pow'] = jnp.array([-1.0])

## 7. Fit

`fitSED` builds the joint likelihood automatically from `model.observations` (one Gaussian likelihood per observation, keyed by `name`). Here we use VI-preconditioned NUTS; pass `sampler='nested'` for nested sampling instead.

In [ ]:
result = fitSED(
    model, observations,
    output_dir='./joint_fit',
    sampler='nuts', vi='tril',
    sampler_kwargs={'num_chains': 4, 'num_samples': 2000},
    rng_key=jax.random.PRNGKey(0),
)

## 8. Posterior and model-vs-data

In [ ]:
samples = result.samples            # dict keyed by parameter name
print('median logmass =', float(np.median(np.asarray(samples['logmass']))))

# Predicted data for a chosen theta (keyed by observation name):
theta = {k: jnp.asarray([float(np.median(np.asarray(samples[k])))])
         for k in ('Z','logmass','diffuse_tau_kc','diffuse_dust_index',
                   'tau_pow','gas_logz','gas_logu','eline_scaling')}
theta['logsfr_ratios'] = jnp.asarray(np.median(np.asarray(samples['logsfr_ratios']), axis=0))

pred = model.predict(theta)         # {'phot': maggies, 'spec': F_nu, 'lines': fluxes}
print({k: np.asarray(v).shape for k, v in pred.items()})

## Consistency checklist for real joint fits

- **Flux systems must agree.** Photometry (maggies), spectrum (`F_ν`) and line fluxes (erg s⁻¹ cm⁻²) must share the absolute normalisation the model produces at `zred`. Mismatched calibration between data sets is the usual cause of a good per-set χ² but a bad joint fit.
- **Don't double-count lines** — mask them out of the continuum spectrum (`spec.mask_lines`).
- **Aperture** — `eline_scaling` (and `calibration`/`noise_floor` for the spectrum) absorb slit-vs-photometry differences.
- **Rest-frame vacuum Å** for the spectrum and line lists; the model handles redshifting to the observed frame.